In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab05.ipynb")

# Lab 5:  Modeling (Misspecification, Empirical Bayes, and Prior Predictive Checking)

Welcome to the fifth Data 102 lab! 

This lab has three goals. You will:

1. Study the sensitivity of Bayesian estimators to errors in the chosen prior model. 

1. Fit the parameters of a prior by maximizing the evidence. 

1. Validate your prior by running a prior predictive check.

You will be asked to complete the code in this lab from scratch. You are welcome to use AI tools to help you code, but are responsible for the correctness of the code, and your own comprehension of its mechanics. You may be asked to replicate code snippets requested here, as pseudo-code, on an exam. Where asked to describe a plot result, or reason with a conclusion, double check any code outputs. You may be asked to provide similar description or reasoning for sample figures/outputs on an exam. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm # for colorbar formatting (optional)
from scipy import optimize, stats
from scipy.special import gammaln

rng = np.random.default_rng(102) # fix random number generator

# Question 1: Sensitivity to Prior Misspecification

This question is closely related to Discussion 5. Please review that question before starting here. Note, since this problem reuses work from Discussion 5, the questions on Discussion 5 will be in scope for your second exam. 

In most Bayesian contexts we do not know our priors exactly. So, it is important to study how robust our inferential procedures are to errors in the prior (that is, to prior misspecification). 

Consider the setting you explored in Discussion 5. You can observe a string of i.i.d. random variables $\{X_i\}_{i=1}^n$ drawn normally with unknown mean $\Theta$ and known variance $\sigma_x^2$. To estimate the mean you adopt the Bayesian model:

$$\Theta \sim \text{Normal}(\mu_{\theta},\sigma_{\theta}^2), \quad \quad \{X_i\}_{i=1}^n |\Theta = \theta \underset{\text{i.i.d}}{\sim} \text{Normal}(\theta,\sigma_x^2)$$

Then, under your model:

$$\Theta|\{X_i = x_i\}_{i=1}^n \sim \text{Normal}\left( a, b \right) $$

where:

$$\begin{aligned}
    & a = (1 - \lambda) \hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) + \lambda \mu_{\theta}, \\
    & \hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) = \frac{1}{n} \sum_{i=1}^n x_i, \\
    & \lambda = \frac{1/\sigma_{\theta}^2}{n/\sigma_x^2 + 1/\sigma_{\theta}^2} \\
    & b = \frac{1}{n/\sigma_x^2 + 1/\sigma_{\theta}^2}
\end{aligned} $$

You adopt the posterior mean as your estimator:

$$\hat{\theta}_{\text{mean}}(\{x_i\}_{i=1}^n) = \mathbb{E}[\Theta|\{X_i = x_i\}_{i=1}^n] =  (1 - \lambda) \hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) + \lambda \mu_{\theta}$$

Suppose that, in truth:

$$\Theta \sim \text{Normal}(0,1). $$

Then your model is misspecified whenever $\mu_{\theta} \neq 0$ or $\sigma_{\theta} \neq 1$. Let's work out how much error we can admit before our posterior mean estimator is less reliable (has a larger risk) than the MLE. 

<!-- BEGIN QUESTION -->

## 1a) Checking the MLE

Confirm that the MLE is the sample average, $\bar{x}_n$:
    
$$\hat{\theta}_{\text{MLE}}(\{x_i\}_{i=1}^n) = \bar{x}_n = \frac{1}{n} \sum_{i=1}^n x_i. $$

> **Solution.**

The likelihood is

$$L(\theta) = \prod_{i=1}^n \frac{1}{\sqrt{2\pi\sigma_x^2}} \exp\left(-\frac{(x_i-\theta)^2}{2\sigma_x^2}\right), \qquad \ell(\theta) = \log L(\theta) = c - \frac{1}{2\sigma_x^2}\sum_{i=1}^n (x_i - \theta)^2 $$

where $c$ is the log normalization constant.

Then

$$\frac{d}{d\theta}\ell(\theta) = \frac{1}{\sigma_x^2}\sum_{i=1}^n (x_i - \theta) = \frac{n}{\sigma_x^2}(\bar x_n - \theta) = 0 \;\Rightarrow\; \theta = \bar x_n.$$

To check that $\bar{x}_n$ maximizes the likelihood, evaluate $\frac{d^2}{d\theta^2} \ell''(\theta) = -n/\sigma_x^2 < 0$, so this stationary point is the unique global maximum. Hence $\hat\theta_{\text{MLE}} = \bar x_n$.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 1b) Generality

It may seem odd that we are starting from a fixed true model. Show that, if, in truth, $\Psi \sim \text{Normal}(m,v) $, we assumed $\Psi \sim \text{Normal}(\mu_{\psi},\sigma_{\psi}^2)$, and observed $\{Y_i\}_{i=1}^n|\Psi = \psi \sim \text{Normal}(\psi,\sigma_y^2)$, then we could recover the case introduced above via the change of coordinates $x = (y - m)/\sqrt{v}$. Apply this change of coordinates to express $\Theta$ as a function of $\Psi$. Show that the true prior for $\Theta$ is $\text{Normal}(0,1)$, and our assumed prior is $\text{Normal}((\mu_{\psi} - m)/\sqrt{v},\sigma_{\psi}^2/v)$. 

> **Solution.**

Since $\Theta$ is the expected value of each $X_i$ we should apply the same coordinate change to $\Theta$ that we applied to $X$. Since $X_i = (Y_i - m)/\sqrt v$, $\Theta = (\Psi - m)/\sqrt v$. Then $\mathbb E[\Theta] = (m - m)/\sqrt v = 0$ and $\text{Var}[\Theta] = v/v = 1$. A linear transformation of any normal random variable is also normal so $\Theta \sim \text{Normal}(0,1)$. 

Applying the same expectation logic, if $\Psi \sim \text{Normal}(\mu_\psi, \sigma_\psi^2)$ then the assumed prior becomes $\text{Normal}\big((\mu_\psi - m)/\sqrt v,\; \sigma_\psi^2/v\big)$.

<!-- END QUESTION -->

So, we can assume, without loss of generality, that the true prior is $\Theta \sim \text{Normal}(0,1)$ and interpret:

1. $\mu_{\theta}$ as the standardized discrepancy in means, $ (\mu_{\psi} - m)/\sqrt{v}$, and
1. $\sigma_{\theta}$ as the ratio of the assumed and true standard deviations $\sigma_{\psi}/\sqrt{v}$. 

## 1c) Bias and Variance in the Estimators

Repeat steps (c) - (f) from Discussion 5 to compute the Bayes risk of the posterior mean estimator and the MLE, averaging over the *true* prior model:

$$R_B(\hat{\theta}) = \mathbb{E}_{\Theta,\{X_i\}_{i=1}^n}[(\hat{\theta}(\{X_i\}_{i=1}^n) - \Theta)^2].  $$

Note, most of this analysis will be the same. However, the posterior mean estimator will now include an additional bias associated with the misspecification error.

<!-- BEGIN QUESTION -->

### 1c (i) Bias

Find the Frequentist and Bayes bias in the MLE and posterior mean estimator, averaging over the *true* prior model, but drawing estimates using the assumed prior.

> **Solution.**

Throughout, $\lambda = \dfrac{1/\sigma_\theta^2}{n/\sigma_x^2 + 1/\sigma_\theta^2}$ uses the **assumed** $\sigma_\theta$. The expectations are over the **true** model, $\Theta \sim \text{Normal}(0,1)$.

**Frequentist bias (given $\Theta = \theta$):** unchanged from discussion 5, because conditioning on $\Theta$ removes the prior:
$$\mathbb E[\hat\theta_{\text{MLE}} - \Theta \mid \Theta = \theta] = 0, \qquad \mathbb E[\hat\theta_{\text{mean}} - \Theta \mid \Theta = \theta] = (1-\lambda)\theta + \lambda\mu_\theta - \theta = \lambda(\mu_\theta - \theta).$$

**Bayes bias (joint, under the true prior):**
$$\mathbb E[\hat\theta_{\text{MLE}} - \Theta] = 0, \qquad \mathbb E[\hat\theta_{\text{mean}} - \Theta] = \mathbb E_\Theta[\lambda(\mu_\theta - \Theta)] = \lambda(\mu_\theta - 0) = \lambda\mu_\theta .$$
This is the new term: the posterior mean is no longer jointly unbiased, because it shrinks toward the wrong center.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 1c (ii) Variance

Find the Frequentist and Bayes variance in the MLE and the posterior mean estimator, averaging over the *true* prior model, but drawing estimates using the assumed prior.

> **Solution.**

**Frequentist variance:** unchanged for the same reason provided above:
$$\text{Var}[\hat\theta_{\text{MLE}} - \Theta \mid \Theta = \theta] = \frac{\sigma_x^2}{n}, \qquad \text{Var}[\hat\theta_{\text{mean}} - \Theta \mid \Theta = \theta] = (1-\lambda)^2\frac{\sigma_x^2}{n}.$$

**Bayes variance (law of total variance, true prior $\text{Var}[\Theta] = 1$):**
$$\text{Var}[\hat\theta_{\text{MLE}} - \Theta] = \mathbb E\left[\tfrac{\sigma_x^2}{n}\right] + \text{Var}[0] = \frac{\sigma_x^2}{n},$$
$$\text{Var}[\hat\theta_{\text{mean}} - \Theta] = (1-\lambda)^2\frac{\sigma_x^2}{n} + \text{Var}_\Theta[\lambda(\mu_\theta - \Theta)] = (1-\lambda)^2\frac{\sigma_x^2}{n} + \lambda^2 .$$
(The variance term is $\lambda^2 \cdot 1$, the **true** prior variance, not $\lambda^2\sigma_\theta^2$.)

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 1c (iii) Risk

Find the Bayes risk for each estimator, averaging over the *true* prior model, but drawing estimates using the assumed prior.

> **Solution.**

$R_B = (\text{Bayes bias})^2 + \text{Bayes variance}$:
$$R_B(\hat\theta_{\text{MLE}}) = \frac{\sigma_x^2}{n}, \qquad R_B(\hat\theta_{\text{mean}}) = \underbrace{(1-\lambda)^2\frac{\sigma_x^2}{n}}_{\text{data noise}} + \underbrace{\lambda^2}_{\text{true prior spread}} + \underbrace{\lambda^2\mu_\theta^2}_{\text{misspecified center}} = (1-\lambda)^2\frac{\sigma_x^2}{n} + \lambda^2(\mu_\theta^2 + 1).$$
Sanity check: with the correct prior ($\mu_\theta = 0$, $\sigma_\theta = 1$) this reduces to the discussion-5 result $(1-\lambda)^2\sigma_x^2/n + \lambda^2\sigma_\theta^2$.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 1d) Risk Comparison

Show that the difference between the Bayes risks for the MLE and the posterior mean estimator is:

$$R_B(\hat{\theta}_{\text{MLE}}) - R_B(\hat{\theta}_{\text{mean}}) = \left(1 - (1 - \lambda)^2 \right) R_B(\hat{\theta}_{\text{MLE}}) - \lambda^2 (\mu_{\theta}^2 + 1), \text{ where } R_B(\hat{\theta}_{\text{MLE}}) = \frac{\sigma_x^2}{n}$$



> **Solution.**

Subtract the two risks from 1c(iii):
$$\Delta R = \frac{\sigma_x^2}{n} - (1-\lambda)^2\frac{\sigma_x^2}{n} - \lambda^2(\mu_\theta^2+1) = \big(1 - (1-\lambda)^2\big)R_B(\hat\theta_{\text{MLE}}) - \lambda^2(\mu_\theta^2+1). \qquad \blacksquare$$

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Call this difference in risks $\Delta R$. The posterior mean estimator is more reliable than the MLE as long as $\Delta R > 0$. It is much more reliable if $\Delta R \gg 0$. It is less reliable if $\Delta R < 0$. 

In the space below, write a function that can compute $\Delta R$ given the parameters:

1. The standardized error in the estimated prior mean: $\mu_{\theta} = (\mu_{\psi} - m)/\sqrt{v}$

1. The ratio of the assumed prior standard deviation and true standard deviation: $\sigma_{\theta} = \sigma_{\psi}/\sqrt{v}$

1. The standard deviation in the sampled data: $\sigma_x$

1. The sample size $n$.

Your code should be vectorized so it can compute $\Delta R$ for many different input parameter combinations efficiently.

In [ ]:
# compute lambda
def shrinkage(sigma_theta, sigma_x, n):
    # lambda = prior precision / (data precision + prior precision)
    prior_prec = 1.0 / np.asarray(sigma_theta, dtype=float)**2
    data_prec = np.asarray(n, dtype=float) / np.asarray(sigma_x, dtype=float)**2
    return prior_prec / (data_prec + prior_prec)


# compute risk for MLE
def risk_mle(sigma_x, n):
    return np.asarray(sigma_x, dtype=float)**2 / np.asarray(n, dtype=float)

# compute risk for posterior mean
def risk_post_mean(mu_theta, sigma_theta, sigma_x, n):
    lam = shrinkage(sigma_theta, sigma_x, n)
    return (1 - lam)**2 * risk_mle(sigma_x, n) + lam**2 * (np.asarray(mu_theta)**2 + 1)

# compute difference in risk between MLE and posterior mean
def delta_R(mu_theta, sigma_theta, sigma_x, n):
    # R_B(MLE) - R_B(posterior mean), averaged over the TRUE prior Theta ~ N(0,1).
    lam = shrinkage(sigma_theta, sigma_x, n)
    R_mle = risk_mle(sigma_x, n)
    return (1 - (1 - lam)**2) * R_mle - lam**2 * (np.asarray(mu_theta)**2 + 1)

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

In the space below, generate three plots, each showing $\Delta R$ as a function of $\mu_{\theta} \in [-5,5]$ and $\sigma_{\theta} \in (0.1,10]$ for $\sigma_x = 3$ and $n = 1$, $n = 10$, and $n = 100$. Use the following display settings:

1. Set your $x$-axis to $\mu_{\theta}$ and your $y$-axis to $\sigma_{\theta}$. Use a log 10 scale for $\sigma_{\theta}$.

1. Display the surface $\Delta R(\mu,\sigma)$ as a heatmap with overlaid contours. Add a colorbar for each plot.

1. Overlay, in bold, the contour corresponding to $\Delta R(\mu,\sigma) = 0$. 

In [ ]:
# set parameter ranges for plots
mu_grid = np.linspace(-5, 5, 501)
sig_grid = np.logspace(-1, 1, 501)            # sigma_theta in [0.1, 10], log-spaced
MU, SIG = np.meshgrid(mu_grid, sig_grid)
sigma_x = 3.0

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), constrained_layout=True)
for ax, n in zip(axes, [1, 10, 100]):
    D = delta_R(MU, SIG, sigma_x, n) # compute difference in risk for current n

    norm = TwoSlopeNorm(vmin=D.min(), vcenter=0.0, vmax=D.max())   # diverging, centered at 0 (for color scale)
    pc = ax.pcolormesh(MU, SIG, D, cmap="RdBu_r", norm=norm, shading="auto") # set color scale
    fig.colorbar(pc, ax=ax, label=r"$\Delta R = R_B(\mathrm{MLE}) - R_B(\mathrm{post.\ mean})$")

    # thin contours: log-spaced negative levels, a few positive levels
    neg = -np.array([0.01, 0.03, 0.1, 0.3, 1, 3, 10, 20])
    neg = neg[neg > D.min()]
    pos = D.max() * np.array([0.25, 0.5, 0.75, 0.95])
    cs = ax.contour(MU, SIG, D, levels=np.sort(np.r_[neg, pos]), colors="k", linewidths=0.6, alpha=0.6)
    ax.clabel(cs, fontsize=7, fmt="%.2g")

    # bold zero contour
    ax.contour(MU, SIG, D, levels=[0.0], colors="k", linewidths=3)

    ax.axhline(1, color="gray", lw=0.8, ls=":")   # correct sigma_theta
    ax.axvline(0, color="gray", lw=0.8, ls=":")   # correct mu_theta

    ax.set_yscale("log")
    ax.set_xlim(-5, 5); ax.set_ylim(0.1, 10)

    ax.set_xlabel(r"$\mu_\theta$ (standardized error in prior mean)")
    ax.set_ylabel(r"$\sigma_\theta$ (assumed / true prior sd)")
    ax.set_title(rf"$\Delta R$, $\sigma_x = 3$, $n = {n}$")
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 1e) Conclusions

Whenever you generate a plot, you should reason with the result. In the space below, describe the plots you generated above, highlighting observations you think would be important to a user who wants to adopt the posterior mean estimator, but is concerned that they will use the wrong prior parameters. Make sure that your description answers the question: "*How large can the errors in the prior parameters be before the MLE is more reliable than the posterior mean estimator?*"

> **Solution.**

There are many possible answers to this part. Here's one. 

The plots show that, for any given $n$, the posterior estimator has:

* a smaller risk if we overestimate the prior variance,
* a larger risk if we significantly underestimate the prior variance,
* is insensitive to the prior mean if the prior variance is large, so larger misspecification in the prior mean are admissible as the presumed prior variance grows
* as $n$ increases the MLE estimator becomes more stable, so the region where the posterior mean estimator is better shrinks. So, if $n$ is large, then the prior must be resolved precisely, otherwise the misspecification errors in the prior induce inadmissible biases.
* The posterior mean estimator can be better than the MLE, and is best when correctly specified. However, its advantage over the MLE shrinks as $n$ grows. 
* When misspecified, the posterior mean may be much worse than the MLE.

**How large can the prior errors be?** In standardized units, the posterior mean is safer than the MLE exactly when
$$\left(\frac{\mu_\psi - m}{\sqrt v}\right)^2 < \frac{\sigma_y^2}{n v} + 2\frac{\sigma_\psi^2}{v} - 1 .$$
Roughly: keep the prior mean within about one true prior sd of the truth, and don't understate the prior variance by more than about a factor of 2. If unsure, err toward a wider prior.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Suppose that you select the correct prior variance, $\sigma_{\theta}^2 = 1$. Find the range of prior means, $\mu_{\theta}$, for which the posterior mean estimator is more reliable than the MLE. Express your answer in terms of $\sigma_x$ and $n$.

> **Solution.**

First, solve for the boundary where the two methods produce the same risk.

Write $r = \sigma_x^2/n$ and $s = \sigma_\theta^2$. Then $\lambda = r/(r+s)$ and $1-(1-\lambda)^2 = \lambda(2-\lambda)$, so
$$\Delta R = \lambda\left[(2-\lambda)\,r - \lambda(\mu_\theta^2+1)\right] = \frac{\lambda\, r}{r+s}\left[\,r + 2s - (\mu_\theta^2 + 1)\right].$$

Since $\lambda r/(r+s) > 0$:
$$\Delta R > 0 \iff \mu_\theta^2 < \frac{\sigma_x^2}{n} + 2\sigma_\theta^2 - 1\$$

If $\sigma_{\theta} = 1$ then $\Delta R > 0 \iff \mu_\theta^2 < 1 + \sigma_x^2/n$, i.e.
$$|\mu_\theta| < \sqrt{1 + \sigma_x^2/n}.$$

For $\sigma_x = 3$: $|\mu_\theta| < \sqrt 10 \approx 3$ ($n=1$), $\sqrt{1.9} \approx 1.4$ ($n=10$), $\sqrt{1.09} \approx 1.04$ ($n=100$). The window shrinks to $|\mu_\theta| = 1$ as $n \to \infty$.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Suppose that you select the correct prior mean, $\mu_{\theta} = 0$. Find the range of prior variances, $\sigma_{\theta}^2$, for which the posterior mean estimator is more reliable than the MLE. Express your answer in terms of $\sigma_x$ and $n$.

> **Solution.**

Set $\mu_\theta = 0$: $\Delta R > 0 \iff 0 < \sigma_x^2/n + 2\sigma_\theta^2 - 1$, i.e.
$$\sigma_\theta^2 > \frac{1 - \sigma_x^2/n}{2}.$$

If $\sigma_x^2/n \ge 1$, this holds for every $\sigma_\theta^2 > 0$. 

For $\sigma_x = 3$: any $\sigma_\theta^2 > 0$ ($n = 1$), $\sigma_\theta^2 > 0.05$ ($n=10$), $\sigma_\theta^2 > 0.455$ ($n = 100$). As $n \to \infty$ the threshold tends to $1/2$: you can understate the prior variance by up to a factor of two. 

There is **no upper bound**.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

You should notice that these answers are different. Given the correct variance, there are upper and lower bounds on the assumed means outside of which the MLE is more reliable. In contrast, given the correct prior mean, the posterior mean estimator is more reliable whenever $\sigma_{\theta}^2$ is greater than a lower bound. In other words, there is no upper bound on the error in the assumed variance past which we should use the MLE. We are safe to use the posterior mean estimator *no matter* how much we overestimate the prior variance. 

Justify this observation conceptually.

> **Solution.**

The prior variance controls **how much** we bias towards the prior mean. The prior mean controls **where** we bias towards.

* **Overstating the variance** ($\sigma_\theta^2$ too large) makes $\lambda$ smaller, so we bias *less* than is optimal. In the limit $\sigma_\theta \to \infty$, $\lambda \to 0$ and the estimator becomes the MLE. When $\mu_{\theta} = 0$ we biasing (weakly) toward the *correct* center, so overestimating $\sigma_\theta^2$ results in an estimator that is more stable than the MLE, but not as stable as it could be using the true prior variance. 
* **Understating the variance** makes the prior overconfident. $\lambda$ gets close to 1, and the estimator ignores the data and returns roughly $\mu_\theta$. Its error is then roughly $\mu_\theta - \Theta$, whose risk is about the true prior variance (1), which can far exceed $\sigma_x^2/n$ once $n$ is large.
* **A wrong mean** adds bias $\lambda\mu_\theta$, which grows without bound in $|\mu_\theta|$ for any fixed $\lambda > 0$. Any fixed amount of reduction in variance is eventually outweighed if you bias toward a point far enough from the truth. 

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

For each $n$, identify the largest value of $\Delta R$. What choice of prior parameters gives the greatest advantage over the MLE? How does the best case $\Delta R$ behave as a function of $n$?

> **Solution.**

The best choice of prior parameters are always the true parameters. The largest value of $R$ is:

* **For $n = 1$:** $\Delta R_{\text{max}} = 8.1$.
* **For $n = 10$:** $\Delta R_{\text{max}} = \approx 0.7368$.
* **For $n = 100$:** $\Delta R_{\text{max}} = \approx 0.0074$.

The advantage of the posterior mean estimator decays rapidly in $n$. 

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

For each $n$, identify the smallest (most negative) value of $\Delta R$. What choice of prior parameters is the worst relative to the MLE? How does the worst case $\Delta R$ behave as a function of $n$?

> **Solution.**

There is no lower bound on $\Delta R$. Simply select the most misspecified prior possible. Set $\sigma_{\theta} = 0$, so that the posterior mean estimator equals $\mu_{\theta}$. Then take $|\mu_{\theta}|$ to infinity to maximize the bias. 

Reading off the plots above, the worst case $\Delta R$, over the plotted range, are about -16. -24, and -20, all at  $|\mu_{\theta}| = \pm 5$ and $\sigma_{\theta} = 0.1$. 

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Compare the best-case $\Delta R$ to the worst-case $\Delta R$. Based on your observations above, should a user, who cannot fit their prior exactly, adopt a Bayesian method when $n$ is small or when $n$ is large? How would your suggested threshold on $n$ change if the data were more or less variable (try re-running your code with $\sigma_x = 2$ and $\sigma_x = 4$)?

> **Solution.**

The Bayes risk for the MLE is never worse than the variance in the MLE, which vanishes as $n$ diverges. So, switching to the posterior mean estimator can only reduce the Bayes risk substantially when the MLE is highly variable. On the other hand, the risk for a misspecified posterior mean estimator may be arbitrarily large if the posterior mean biases towards an arbitrarily wrong prior mean. The larger the error in the prior mean, the larger the Bayes risk in the misspecified estimator. 

So, there is unlimited possible downside to using the posterior mean estimator, but limited upside. The posterior mean estimator should only be accepted when the MLE is too variable (e.g. small $n$) and when the prior mean can be guessed to reasonable accuracy (e.g. to an error on the scale of, or smaller than, the standard deviation in the MLE). So, a user should adopt a Bayesian estimator when $n$ is too small. The threshold for small enough should increase as $\sigma_x$ increases.


<!-- END QUESTION -->

# Question 2: Fitting Prior Parameters

Consider the following conjugate Bayesian model:

1. $\{\Lambda_i\}_{i=1}^n$ are drawn i.i.d. from a $\text{Gamma}(a,b)$ distribution:

    $$ \Lambda_i \geq 0, \quad f_{\Lambda_i}(\lambda) = \frac{b^a}{\Gamma(a)} \lambda^{a - 1} e^{-b \lambda} $$

1. $\{X_i|\Lambda_i = \lambda \}_{i=1}^n$ are drawn independently from a $\text{Poisson}(\lambda)$ distribution

    $$X_i \in \{0,1,2,...\}, \quad \text{Pr}(X_i = x|\Lambda_i = \lambda) = \frac{\lambda^x}{x!} e^{-\lambda}$$

You can observe the counts $\{X_i\}_{i=1}^n$, but not $\{\Lambda_i\}_{i=1}^n$. You would like to use your observations to estimate the prior parameters $a, b$ so that, given a new observation $X_{n+1} = x_{n+1}$, you could form posterior estimates of $\Lambda_{n+1}$. 

<!-- BEGIN QUESTION -->

## 2a) Posterior Inference

Suppose, for now, that you had already fit for $a$ and $b$ (see below), and we observed $X_{n+1} = x_{n+1}$. Find the posterior distribution for $\Lambda_{n+1}|X_{n+1} = x_{n+1}$ and provide a formula for the maximum a posteriori estimator for $\Lambda_{n+1}$. 

> **Solution.**

By Bayes' rule (dropping factors that don't depend on $\lambda$):

$$f_{\Lambda_{n+1}\mid X_{n+1} = x}(\lambda) \propto \lambda^{a-1}e^{-b\lambda} \cdot \lambda^{x}e^{-\lambda} = \lambda^{(a + x) - 1}e^{-(b+1)\lambda},$$

which is the functional form of a gamma density. So

$$\Lambda_{n+1}\mid X_{n+1} = x_{n+1} \sim \text{Gamma}(a + x_{n+1},\; b + 1).$$

Then, the MAP can be recovered from the formula for the mode of a gamma:

$$\hat\lambda_{\text{MAP}} = \begin{cases} & \frac{a + x_{n+1} - 1}{b + 1} \quad & \text{if } a + x_{n+1} \ge 1, \\ & \hat\lambda_{\text{MAP}} = 0 & \text{otherwise} \end{cases}$$

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 2b) The Evidence

In a Bayesian model the evidence is the marginal distribution of the data, marginalizing over the unknowns. 

Find the marginal distribution of $X_i$, marginalizing over $\Lambda_i$. Express your answer by completing the probability mass function:

$$\text{Pr}(X_i = x_i) = \underline{~~~~~~~~~~~~~~~~~~~~~~~~~~~~~} $$

To close your integral you may use the value of the gamma integral suggested by the normalization factor of the gamma distribution:

$$\int_{\lambda = 0}^{\infty} \lambda^{c - 1} e^{-d \lambda} d\lambda = \frac{\Gamma(c)}{d^c}. $$


> **Solution.**

$$\Pr(X_i = x) = \int_0^\infty \frac{\lambda^x e^{-\lambda}}{x!}\,\frac{b^a}{\Gamma(a)}\lambda^{a-1}e^{-b\lambda}\,d\lambda = \frac{b^a}{\Gamma(a)\,x!}\int_0^\infty \lambda^{a + x - 1}e^{-(b+1)\lambda}d\lambda = \frac{b^a}{\Gamma(a)\,x!}\cdot\frac{\Gamma(a+x)}{(b+1)^{a+x}} .$$
So
$$\Pr(X_i = x) = \frac{\Gamma(a + x)}{\Gamma(a)\,x!}\left(\frac{b}{b+1}\right)^{a}\left(\frac{1}{b+1}\right)^{x}, \quad x = 0, 1, 2, \ldots$$


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

This distribution is also called the *prior predictive distribution* since it is the distribution we would use to predict the value of a new data point before observing any other data. Notice that its parameters are the parameters of the prior.

So, the prior predictive distribution, evaluated at an observed outcome, is, as a function of the prior parameters, the likelihood function for the prior parameters. It follows that selecting the prior parameters that maximize the evidence is equivalent to selecting the maximum likelihood *prior* parameters. The prior parameters that maximize the evidence are the prior parameters that, if true, would make the observed data most likely!

The argument above justifies the name "*evidence*". The larger the prior predictive distribution, evaluated at the observation, the more evidence we have in favor of our prior model (here, our choice of prior parameters).

In the space below, complete the expressions:

$$\text{Pr}(X_1 = x_1,X_2 = x_2,...,X_n = x_n) = \underline{~~~~~~~~~~~~~~~~~~~~~~~~~~~~~}, \quad  \log(\text{Pr}(X_1 = x_1,X_2 = x_2,...,X_n = x_n)) = \underline{~~~~~~~~~~~~~~~~~~~~~~~~~~~~~}$$

> **Solution.**

The pairs $(\Lambda_i, X_i)$ are independent across $i$ (each count has its own rate), so the $X_i$ are i.i.d. from the prior predictive and the evidence factorizes:

$$\Pr(X_1 = x_1, \ldots, X_n = x_n) = \prod_{i=1}^n \frac{\Gamma(a + x_i)}{\Gamma(a)\,x_i!}\left(\frac{b}{b+1}\right)^{a}\left(\frac{1}{b+1}\right)^{x_i},$$

$$\log \Pr(X_1 = x_1, \ldots, X_n = x_n) = \sum_{i=1}^n \log\Gamma(a + x_i) - n\log\Gamma(a) - \sum_{i=1}^n \log(x_i!) + n a\log b - \Big(n a + \sum_{i=1}^n x_i\Big)\log(b+1).$$

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

In the code cell below, write a function which can compute $\log(\text{Pr}(X_1 = x_1,X_2 = x_2,...,X_n = x_n))$ as a function of the observations $\{x_i\}_{i=1}^n$, and prior parameters $a$ and $b$.

In [ ]:
def log_evidence(x, a, b):
    # log Pr(X_1 = x_1, ..., X_n = x_n) under the Gamma(shape a, rate b)-Poisson model.
    # Uses gammaln for numerical stability (log Gamma, and log x! = gammaln(x + 1)), and log1p for log(b + 1).
    x = np.asarray(x, dtype=float)
    n, sx = x.size, x.sum()
    return (gammaln(a + x).sum() - n * gammaln(a) - gammaln(x + 1).sum()
            + n * a * np.log(b) - (n * a + sx) * np.log1p(b))

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 2c) Empirical Bayes

Suppose that you observe the string of counts (sorted from smallest to largest):

$$\{X_i\}_{i=1}^{20} = \{0,3,6,6,8,12,13,13,14,15,16,20,20,21,24,24,27,27,30,40\} $$

Write code below that finds the parameters $a$ and $b$ which maximize the likelihood of this observation. Note: it will be more stable to maximize the log evidence instead of the evidence itself. You may call a pre-built optimizer.

In [ ]:
x_obs = np.array([0, 3, 6, 6, 8, 12, 13, 13, 14, 15, 16, 20, 20, 21, 24, 24, 27, 27, 30, 40])

def fit_prior(x, init=None):
    # Maximize the log evidence over (a, b). Optimize over (log a, log b) so the
    # positivity constraints are automatic. The bounds guard against near-Poisson
    # samples (var <= mean), where the maximizer runs off to a, b -> infinity.
    x = np.asarray(x, dtype=float)                     # format input list of counts
    if init is None:                                   # initialize with method-of-moments estimate
        m, v = x.mean(), x.var(ddof=1)                 # sample mean and variance
        b0 = m / (v - m) if v > m else 1.0  
        init = np.log([max(m * b0, 1e-3), b0])         # initial guess for log(a) and log(b) chosen so that the moments of the prior predictive match the sample moments
    res = optimize.minimize(lambda p: -log_evidence(x, *np.exp(p)), init,
                            method="L-BFGS-B", bounds=[(-10, 15), (-15, 15)]) # optimize using l-bfgs-b
    a_hat, b_hat = np.exp(res.x)                       # transform back from log(a), log(b) to a, b
    return a_hat, b_hat, -res.fun

a_hat, b_hat, ll_hat = fit_prior(x_obs)
print(f"a_hat = {a_hat:.4f}, b_hat = {b_hat:.4f}, max log evidence = {ll_hat:.4f}")
print(f"implied prior on rates: mean a/b = {a_hat/b_hat:.3f} (sample mean {x_obs.mean()}), sd sqrt(a)/b = {np.sqrt(a_hat)/b_hat:.3f}")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Report your best fit parameters below:

> **Solution.**

The best fit prior parameters are $a = 2.507$ and $b = 0.148$. 

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 2d) Model Validation (Prior Predictive Checking)

Next, we should check whether our best fit model could have plausibly generated the observed sample. 

In the space below, write code that samples from the Bayesian model for arbitrary prior parameters $a$ and $b$. Your code should produce sample strings of length $n$ for any integer $n$.

*Note: $b$ is a rate parameter. `scipy.stats.gamma` and `np.random.gamma` are parameterized by scale, so pass `scale=1/b`.*

In [ ]:
def sample_model(a, b, n, size=None, rng=rng):
    # Draw from the joint model: Lambda_i ~ Gamma(shape=a, rate=b), X_i | Lambda_i ~ Poisson(Lambda_i).
    # Returns an array of shape (n,) if size is None, else (size, n): `size` strings of length n.
    shape = (n,) if size is None else (size, n)
    lam = rng.gamma(shape=a, scale=1.0 / b, size=shape)   # NOTE: numpy uses scale = 1/rate
    return rng.poisson(lam)

<!-- END QUESTION -->

Run an NHST to determine whether your best fit model could have plausibly generated the observed data. To run your NHST:

1. Adopt a one-sample [Kolmogorov-Smirnov test](https://en.wikipedia.org/wiki/Kolmogorov%E2%80%93Smirnov_test). The KS test applies since, under the joint model, the samples $\{X_i\}_{i=1}^n$ are drawn i.i.d. from the prior predictive distribution. Your KS statistic should equal the maximum absolute difference between the empirical cumulative distribution function of the observed samples and your best fit cumulative distribution function. 

1. Estimate the $p$-value for your observed KS statistic by sampling 10,000 samples of length 20 from your best fit model, then for each sampled data set:

    - refit for the best $a, b$ 
    
    - compute the associated KS statistic
    
    - then count the fraction that are larger than or equal to the observed KS statistic.

<!-- BEGIN QUESTION -->

In the space below, explain why we should refit $a, b$ for each simulated dataset. If you are unsure, repeat the process with and without refitting and compare $p$-values. 

> **Solution.**

By refitting on each stage we fully imitate the process from data to test statistic. Our test statistic is the KS statistic applied to the model that *best fits* the observed data. So, the definition of the test statistic includes the fitting stage.

If we did not include the fitting stage, then, when sampling, we would compute test statistics against a fixed model. The fixed model will typically have a larger KS from a sample collection than the model chosen to best fit each sample collection. So, if we don't refit, the sampled KS values will be too large, so, when comparing to an observed KS stat, which would require larger observed KS stats for rejection. This would lead to an overly permissive test that might fail to reject models that should be rejected. 

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Prepare your code below:

In [ ]:
# Hypothetical (best-fit) CDF: cumulative sum of the negative-binomial pmf derived in 2b
def prior_predictive_logpmf(k, a, b):
    k = np.asarray(k, dtype=float)
    return gammaln(a + k) - gammaln(a) - gammaln(k + 1) + a * np.log(b / (b + 1)) - k * np.log1p(b)

def prior_predictive_cdf(k, a, b):
    # P(X <= k) for integer k (array OK). Returns 0 for k < 0.
    k = np.asarray(k)
    kmax = int(max(k.max(), 0))
    cdf_table = np.cumsum(np.exp(prior_predictive_logpmf(np.arange(kmax + 1), a, b)))
    return np.where(k < 0, 0.0, cdf_table[np.clip(k, 0, kmax)])

In [ ]:
# KS statistic: sup_x |F_n(x) - F(x)|. Both CDFs are right-continuous step functions that only
# jump at integers, so the sup over the real line equals the max over integers 0..max(sample).
# (Below 0 both are 0. Above the sample max, F_n = 1 and 1 - F(x) only decreases.)
def ks_statistic(samples, a, b):
    # samples: shape (n,) or (reps, n). Returns one KS statistic per row (vectorized).
    samples = np.atleast_2d(samples)
    grid = np.arange(samples.max() + 1)
    F = prior_predictive_cdf(grid, a, b)                       # (K,)
    ecdf = (samples[:, :, None] <= grid).mean(axis=1)          # (reps, K)
    return np.abs(ecdf - F).max(axis=1)

ks_obs = ks_statistic(x_obs, a_hat, b_hat)[0]
print("observed KS statistic:", round(ks_obs, 4))

# visual: empirical vs fitted CDF
grid = np.arange(0, 61)
plt.figure(figsize=(6, 4))
plt.step(grid, (x_obs[:, None] <= grid).mean(0), where="post", label="empirical CDF (n = 20)")
plt.step(grid, prior_predictive_cdf(grid, a_hat, b_hat), where="post", label="fitted prior predictive CDF")
plt.xlabel("x"); plt.ylabel("CDF"); plt.legend(); plt.title(f"KS = {ks_obs:.3f}"); plt.show()

In [ ]:
def ks_pvalue(x, reps=10_000, rng=rng):
    a0, b0, _ = fit_prior(x)
    ks0 = ks_statistic(x, a0, b0)[0]
    sims = sample_model(a0, b0, len(x), size=reps, rng=rng)
    ks_sim = np.empty(reps)
    for r in range(reps):
        a_r, b_r, _ = fit_prior(sims[r])
        ks_sim[r] = ks_statistic(sims[r], a_r, b_r)[0]
    return np.mean(ks_sim >= ks0), ks0

p_value, ks0 = ks_pvalue(x_obs)
print(f"KS p-value = {p_value:.4f}")

> **Solution.**

The observed KS statistic is about $0.126$. The estimated p-value is about $0.36$. In both cases, we fail to reject, so the model has passed our validation test. 

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

If your estimated $p$-value was larger than 0.05, can you conclude that your model is correct? If not, what is the correct interpretation of your prior predictive check?

> **Solution.**

**No.** Failing to reject is not evidence that the model is true. The correct reading is: *the observed data are not surprising under the fitted model, as measured by the KS statistic*. In other words, the fitted model **could plausibly** have generated data like ours, and this check found no evidence against it.

Reasons this is weak support:
* **Low power:** with $n = 20$, the KS test only detects large departures. Many other priors on $\Lambda_i$ (log-normal, …) would also pass.
* **The KS statistic only looks at one thing,** the maximum CDF gap. Other summaries (number of zeros, tail behaviour) might reveal problems it misses.
* The check validates the **marginal** (prior predictive) distribution of $X$. Different priors on $\Lambda$ can produce similar marginals, so it cannot pin down the prior itself.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

To show that this validation check is capable of falsifying models, either:

1. Vary the prior parameters until you find a pair where the test fails, or

1. Generate hypothetical data under a different model for $\Lambda_i$ that is far from Gamma (e.g. $\Lambda_i \sim 1 + 100 \times \text{Bern}(0.5)$). You may generate hypothetical data with $n > 20$ so that your test has higher power. Note: if you generate new data, you should refit for $a$ and $b$.

> **Solution.**

* **Option 1:** moving the prior mean to 10 or 30 (with the fitted shape) is rejected decisively (p < 0.01). But a prior with the *right mean* and far too little spread is **not** rejected at $n = 20$ (p ≈ 0.1). This holds even for the extreme case that is essentially a Poisson(16.95), even though the data's variance (≈100) is about 6× the Poisson variance. The KS test at $n = 20$ is sensitive to location but weak against wrong spread.
* **Option 2:** with $n = 200$ draws from the bimodal rate model, the refit Gamma–Poisson is rejected (p ≈ 0). 

So the check *can* falsify clearly wrong models.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

Comment on the power of this test. Given only twenty samples, can we sharply resolve the prior, or, can you find a pair of highly distinct priors that both pass the test?

> **Solution.**

As above, the test can reject models, but is weak since the sample size is small and the KS test is relatively insensitive to errors in the prior spread. The examples with the correct mean but wrong variance all pass, despite dramatic differences in the prior SD. 

<!-- END QUESTION -->

## Well done!
You've reached the end of the lab! Make sure you double check your work and make sure that you've answered all the written portions of the lab.

**Run the last cell to generate a zip file of your lab submission.**

To submit your lab to Pensieve, submit the zip file to Lab 5 Code.

In [ ]:
import matplotlib.image as mpimg

img = mpimg.imread('baby_rock_hyrax.jpg')
imgplot = plt.imshow(img)
imgplot.axes.get_xaxis().set_visible(False)
imgplot.axes.get_yaxis().set_visible(False)
plt.show()

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

In [ ]:
grader.export(pdf=False, force_save=True)